# 4장 01. 운영 로그 읽기

이 notebook은 JSONL 로그를 읽고 요청 단위 필드를 확인합니다.


## 기본 개념: structured log와 request trace

Structured log는 사람이 읽는 문장 로그가 아니라, key-value 형태로 남기는 운영 기록입니다. MLOps에서는 모델 응답을 나중에 다시 추적할 수 있어야 하므로 `request_id`, `model_version`, `score`, `threshold`, `prediction` 같은 필드가 중요합니다.

`request_id`는 단일 요청을 다시 찾는 키입니다. `trace_id`는 여러 시스템을 지나간 처리 흐름을 묶는 키입니다. API gateway, model server, feature service, downstream system이 나뉘면 trace id가 없을 때 원인 추적이 끊깁니다.

운영 로그는 metric보다 상세하지만, 전체 추세를 바로 보여 주지는 않습니다. 그래서 로그는 대표 요청을 확인하고, metric은 변화 규모를 확인하는 식으로 함께 사용합니다.

| 로그 필드 | 의미 | MLOps에서 쓰는 곳 |
| --- | --- | --- |
| `request_id` | 요청 단위 식별자 | incident 조사, 재현 요청 |
| `trace_id` | 처리 흐름 식별자 | 분산 tracing, 원인 추적 |
| `model_version` | 응답한 모델 버전 | 배포 전후 비교 |
| `score`, `threshold` | 예측 전후 조건 | 설정 변경과 출력 변화 분리 |
| `validation_failure` | 입력 계약 실패 여부 | client/source system 문제 분리 |

이 노트북에서는 로그 수집 플랫폼을 운영하지 않습니다. 준비된 JSONL을 읽고, 운영 관측에 필요한 필드가 어떤 모양으로 남는지 확인합니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
import json
from pathlib import Path

path = Path("artifacts/logs/chapter_04_anomaly_events.jsonl")
if not path.exists():
    path = Path("../artifacts/logs/chapter_04_anomaly_events.jsonl")
if not path.exists():
    path = Path("../../artifacts/logs/chapter_04_anomaly_events.jsonl")

lines = path.read_text().splitlines()[:20]
events = [json.loads(line) for line in lines]
print("path:", path)
print("events:", len(events))


## 1. 로그를 표로 보기

먼저 요청 ID, 모델 버전, score, prediction, 지연 시간, 검증 실패 여부만 봅니다.


In [ ]:
log_table = pd.DataFrame(events)
columns = ["request_id", "model_version", "score", "prediction", "latency_ms", "validation_failure"]
log_table[columns].head(10)
